# Model Evaluation

In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. Load dataset
df = pd.read_csv("cardio_train.csv", sep=";")

# 2. SAME preprocessing used in Task 2
df["age"] = df["age"] / 365

# Check this BEFORE creating X
print("Age after preprocessing:")
print(df["age"].head())

# 3. Load saved model and scaler
model = joblib.load("Cardio_Prediction_Model.pkl")
scaler = joblib.load("Cardio_Scaler.pkl")

# 4. Create X and y AFTER preprocessing
X = df.drop(["cardio", "id"], axis=1)
y = df["cardio"]

# 5. Same train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# 6. Scale the same columns
columns_to_scale = ["age", "height", "weight", "ap_hi", "ap_lo"]

X_train_scaled = X_train.copy()

X_train_scaled[columns_to_scale] = scaler.transform(
    X_train[columns_to_scale]
)

X_test_scaled = X_test.copy()

X_test_scaled[columns_to_scale] = scaler.transform(
    X_test[columns_to_scale]
)

# 7. Check the exact row
print("\nRaw age:")
print(X_test.loc[46730, "age"])

print("\nScaled age:")
print(X_test_scaled.loc[46730, "age"])

# 8. Predict
y_pred = model.predict(X_test_scaled)

# 9. Evaluation
print("\nModel Evaluation")
print("-------------------------")

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred):.4f}")

print("\nPredicted distribution:")
print(pd.Series(y_pred).value_counts())

Age after preprocessing:
0    50.391781
1    55.419178
2    51.663014
3    48.282192
4    47.873973
Name: age, dtype: float64

Raw age:
59.64383561643836

Scaled age:
0.9334607207490939

Model Evaluation
-------------------------
Accuracy : 0.7233
Precision: 0.7455
Recall   : 0.6795
F1-score : 0.7110

Predicted distribution:
0    7608
1    6392
Name: count, dtype: int64


# Check Overfitting / Underfitting

In [3]:
# Predictions on training data
y_train_pred = model.predict(X_train_scaled)

# Training accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)

# Test accuracy
test_accuracy = accuracy_score(y_test, y_pred)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy    : {test_accuracy:.4f}")

Training Accuracy: 0.7192
Test Accuracy    : 0.7233


# Cross-Validation (5-Fold)  OR Bootstrap

In [4]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

# Create a fresh Logistic Regression model
cv_model = LogisticRegression(max_iter=1000)

# 5-Fold Cross-Validation
cv_scores = cross_val_score(
    cv_model,
    X_train_scaled,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("5-Fold Cross-Validation Scores:")
for i, score in enumerate(cv_scores, 1):
    print(f"Fold {i}: {score:.4f}")

print(f"\nAverage CV Accuracy: {cv_scores.mean():.4f}")
print(f"CV Score Spread (Std): {cv_scores.std():.4f}")

5-Fold Cross-Validation Scores:
Fold 1: 0.7201
Fold 2: 0.7217
Fold 3: 0.7167
Fold 4: 0.7164
Fold 5: 0.7187

Average CV Accuracy: 0.7187
CV Score Spread (Std): 0.0020


# Compare All Models

In [5]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Dictionary to store results
model_results = {}

# --------------------------------------------------
# 1. Logistic Regression
# --------------------------------------------------

logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_scaled, y_train)

y_pred_lr = logistic_model.predict(X_test_scaled)

model_results["Logistic Regression"] = [
    accuracy_score(y_test, y_pred_lr),
    precision_score(y_test, y_pred_lr),
    recall_score(y_test, y_pred_lr),
    f1_score(y_test, y_pred_lr)
]


# --------------------------------------------------
# 2. Random Forest
# --------------------------------------------------

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)

model_results["Random Forest"] = [
    accuracy_score(y_test, y_pred_rf),
    precision_score(y_test, y_pred_rf),
    recall_score(y_test, y_pred_rf),
    f1_score(y_test, y_pred_rf)
]


# --------------------------------------------------
# 3. AdaBoost
# --------------------------------------------------

ada_model = AdaBoostClassifier(
    n_estimators=100,
    random_state=42
)

ada_model.fit(X_train_scaled, y_train)

y_pred_ada = ada_model.predict(X_test_scaled)

model_results["AdaBoost"] = [
    accuracy_score(y_test, y_pred_ada),
    precision_score(y_test, y_pred_ada),
    recall_score(y_test, y_pred_ada),
    f1_score(y_test, y_pred_ada)
]


# --------------------------------------------------
# Display comparison
# --------------------------------------------------

comparison_df = pd.DataFrame(
    model_results,
    index=["Accuracy", "Precision", "Recall", "F1-score"]
).T

print(comparison_df.round(4))

                     Accuracy  Precision  Recall  F1-score
Logistic Regression    0.7233     0.7455  0.6795    0.7110
Random Forest          0.7129     0.7183  0.7022    0.7102
AdaBoost               0.7311     0.7735  0.6547    0.7092


# Hyperparameter Tuning

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [100, 150],
    "max_depth": [10, 20]
}

grid_search = GridSearchCV(
    rf,
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)

Best Parameters: {'max_depth': 10, 'n_estimators': 100}
Best CV Accuracy: 0.7343214162844244


In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Get the best tuned model
best_rf = grid_search.best_estimator_

# Predict on test data
y_pred_tuned_rf = best_rf.predict(X_test_scaled)

# Calculate evaluation metrics
tuned_accuracy = accuracy_score(y_test, y_pred_tuned_rf)
tuned_precision = precision_score(y_test, y_pred_tuned_rf)
tuned_recall = recall_score(y_test, y_pred_tuned_rf)
tuned_f1 = f1_score(y_test, y_pred_tuned_rf)

print("Tuned Random Forest Results")
print("---------------------------")
print(f"Accuracy : {tuned_accuracy:.4f}")
print(f"Precision: {tuned_precision:.4f}")
print(f"Recall   : {tuned_recall:.4f}")
print(f"F1-score : {tuned_f1:.4f}")

Tuned Random Forest Results
---------------------------
Accuracy : 0.7396
Precision: 0.7680
Recall   : 0.6880
F1-score : 0.7258


# Try Advanced Models